# TATN 半监督迁移学习阶段（复现 Table IV/V）

**实验设置**（论文 Section IV.D）：
- 每次实验从 9 条 drive cycle 中留 1 条作为测试集（留一法）
- 其余 8 条中：**7 条无标签**（用源域模型生成伪标签），**1 条半标签**（只用前 50% 时间步的真实标签）
- 使用 Step 7 保存的源域预训练模型作为初始化

**运行前**：Runtime → Change runtime type → GPU

## Step 1：挂载 Google Drive，设置路径

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os

DRIVE_RESULTS    = '/content/drive/MyDrive/Research/mining review/TATN/results'
DRIVE_SRC_MODELS = os.path.join(DRIVE_RESULTS, 'source_models')
DRIVE_TRANSFER   = os.path.join(DRIVE_RESULTS, 'transfer_semisup')
WORK_DIR         = '/content/TATN'

os.makedirs(DRIVE_TRANSFER, exist_ok=True)
print('Drive 已挂载')
print('源域模型目录：', DRIVE_SRC_MODELS)
print('迁移结果目录：', DRIVE_TRANSFER)
print('预训练模型列表：', [f for f in os.listdir(DRIVE_SRC_MODELS) if f.endswith('.pt')] if os.path.exists(DRIVE_SRC_MODELS) else '(目录不存在)')

## Step 2：拉取最新代码

In [ ]:
import shutil

# ⚠️ 修改为你自己 fork 的 repo URL
REPO_URL = 'https://github.com/zhangcastle/TATN.git'

os.chdir('/content')
if os.path.exists(WORK_DIR):
    shutil.rmtree(WORK_DIR)
os.system(f'git clone {REPO_URL} {WORK_DIR} -q')
os.chdir(WORK_DIR)
print('工作目录：', os.getcwd())
print('代码文件：', os.listdir('.'))

## Step 3：安装依赖

In [ ]:
!pip install scipy tqdm scikit-learn -q
import torch
print('PyTorch:', torch.__version__)
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

## Step 4：归一化数据处理

从原始 Drive 数据重新处理归一化数据（与 pretrain_colab Step 5 完全相同的逻辑）。

In [ ]:
import scipy.io as sio
import numpy as np
import os

DRIVE_RAW_DATA = '/content/drive/MyDrive/Research/mining review/TATN/Panasonic 18650PF Data'

def find_drive_cycles_dir(base, candidates):
    for c in candidates:
        p = os.path.join(base, c)
        if os.path.isdir(p):
            return p
    return None

temp_raw = {
    '25':  find_drive_cycles_dir(DRIVE_RAW_DATA, [
               'Panasonic 18650PF Data/25degC/Drive cycles',
               'Panasonic 18650PF Data/25degC/Drive Cycles',
               '25degC/Drive cycles', '25degC/Drive Cycles']),
    '10':  find_drive_cycles_dir(DRIVE_RAW_DATA, [
               '10degC/Drive Cycles', '10degC/Drive cycles',
               'Panasonic 18650PF Data/10degC/Drive Cycles']),
    '0':   find_drive_cycles_dir(DRIVE_RAW_DATA, [
               '0degC/Drive cycles', '0degC/Drive Cycles',
               'Panasonic 18650PF Data/0degC/Drive cycles']),
    'n10': find_drive_cycles_dir(DRIVE_RAW_DATA, [
               '-10degC/Drive Cycles', '-10degC/Drive cycles',
               'Panasonic 18650PF Data/-10degC/Drive Cycles']),
    'n20': find_drive_cycles_dir(DRIVE_RAW_DATA, [
               '-20degC/Drive cycles', '-20degC/Drive Cycles',
               'Panasonic 18650PF Data/-20degC/Drive cycles']),
}
for t, p in temp_raw.items():
    print(f'  {t}: {p if p else "[未找到]"}')

In [ ]:
CYCLE_KW = {
    'Cycle_1': ['Cycle_1','Cycle1'], 'Cycle_2': ['Cycle_2','Cycle2'],
    'Cycle_3': ['Cycle_3','Cycle3'], 'Cycle_4': ['Cycle_4','Cycle4'],
    'NN':      ['_NN_','_NN.'],      'US06':    ['US06'],
    'HWFET':   ['HWFET','HWFTa','HWFTb'],
    'LA92':    ['LA92'],             'UDDS':    ['UDDS'],
}

def match_cycle(fname):
    matched = [c for c, kws in CYCLE_KW.items() if any(kw in fname for kw in kws)]
    return matched[0] if len(matched) == 1 else ('SKIP' if len(matched) > 1 else None)

def read_mat_file(fpath):
    data = sio.loadmat(fpath)
    if 'meas' in data:
        m = data['meas']
        return (m['Time'][0][0].flatten(), m['Current'][0][0].flatten(),
                m['Voltage'][0][0].flatten(), m['Battery_Temp_degC'][0][0].flatten(),
                m['Ah'][0][0].flatten())
    return (data['time'].flatten(), data['current'].flatten(),
            data['voltage'].flatten(), data['temp'].flatten(), data['ah'].flatten())

def process_temp(raw_path, temp_label, out_base, interval=10):
    out_path = os.path.join(out_base, temp_label)
    os.makedirs(out_path, exist_ok=True)
    mat_files = sorted([f for f in os.listdir(raw_path) if f.endswith('.mat')])
    all_data = {}
    for fname in mat_files:
        cycle = match_cycle(fname)
        if cycle is None or cycle == 'SKIP':
            if cycle == 'SKIP': print(f'  [跳过合并] {fname}')
            continue
        t, cur, vol, tmp, ah = read_mat_file(os.path.join(raw_path, fname))
        t2=t[::interval]; cur=cur[::interval]; vol=vol[::interval]
        tmp=tmp[::interval]; ah=ah[::interval]
        idx = next((i for i in range(len(t2)-1) if t2[i+1]-t2[i] < 2), 0)
        seg = (cur[idx:], vol[idx:], tmp[idx:], ah[idx:])
        if cycle in all_data:
            all_data[cycle] = tuple(np.concatenate([all_data[cycle][i], seg[i]]) for i in range(4))
            print(f'  [{temp_label}] 合并: {fname} -> {cycle}')
        else:
            all_data[cycle] = seg
            print(f'  [{temp_label}] 读取: {fname} -> {cycle}  n={len(seg[0])}')
    if not all_data: print(f'[{temp_label}] 无数据'); return
    def grange(i): v=np.concatenate([d[i] for d in all_data.values()]); return v.min(),v.max()
    ranges = [grange(i) for i in range(4)]
    norm = lambda x,r: (x-r[0])/(r[1]-r[0])
    for cycle,(cur,vol,tmp,ah) in all_data.items():
        cn,vn,tn,an = norm(cur,ranges[0]),norm(vol,ranges[1]),norm(tmp,ranges[2]),norm(ah,ranges[3])
        sio.savemat(os.path.join(out_path, f'{temp_label}{cycle}.mat'),
            {'current':cn.reshape(-1,1),'voltage':vn.reshape(-1,1),
             'temp':tn.reshape(-1,1),'ah':an.reshape(-1,1)})
    print(f'[{temp_label}] done -> {out_path}\n')

OUTPUT_BASE = os.path.join(WORK_DIR, 'normalized_data', 'Pan')
for temp_label, raw_path in temp_raw.items():
    print(f'========== {temp_label}°C ==========')
    if raw_path:
        process_temp(raw_path, temp_label, OUTPUT_BASE)
    else:
        print('路径未找到，跳过\n')
print('全部处理完成')

## Step 5：导入模块，配置参数

In [ ]:
import sys, importlib, time, math
import numpy as np
import scipy.io as sio
import torch
import torch.nn as nn
import torch.optim as optim

os.chdir(WORK_DIR)
sys.path.insert(0, WORK_DIR)
for m in ['models', 'mydata', 'train', 'utils', 'pretrain', 'pseudo']:
    if m in sys.modules:
        importlib.reload(sys.modules[m])

from models import conv, lstm, fc, regression, Discriminator, load_saved_model, init_seed
from mydata import Pan_all_set
from train import train as transfer_train
import pseudo

# ===================== 实验配置 =====================
DEVICE        = 'cuda:0' if torch.cuda.is_available() else 'cpu'
SOURCE_TEMP   = '10'           # 源域温度（对应论文 Table IV：10°C → 其他）
TARGET_TEMPS  = ['25', '0', 'n10', 'n20']  # 目标域温度列表

EPOCHS        = 100            # 每个 fold 训练轮数（过多会导致模型偏离预训练权重）
EVAL_INTERVAL = 50             # 每隔多少 epoch 做详细日志
BATCH_SIZE    = 6
LR            = 1e-5           # 小学习率，避免破坏预训练模型
LAMB1         = 1.0            # 监督损失权重（target regression)
LAMB2         = 0.1            # 对抗损失权重
LAMB3         = 0.1            # MMD 损失权重
LABEL_RATIO   = 0.5            # 有标签 cycle 使用的标签比例（前 50%）
SEED          = 0

SOURCE_DATA_PATH = os.path.join(WORK_DIR, 'normalized_data', 'Pan') + '/'
TARGET_DATA_PATH = os.path.join(WORK_DIR, 'normalized_data', 'Pan') + '/'
SOURCE_MODEL_PATH = os.path.join(DRIVE_SRC_MODELS, f'pre-{SOURCE_TEMP}.pt')

ALL_CYCLES = Pan_all_set  # ['Cycle_1.mat', ..., 'UDDS.mat']

print(f'Device: {DEVICE}')
print(f'Source: {SOURCE_TEMP}°C → Targets: {TARGET_TEMPS}')
print(f'Epochs: {EPOCHS}, λ1={LAMB1}, λ2={LAMB2}, λ3={LAMB3}')
print(f'Source model: {SOURCE_MODEL_PATH}')
print(f'Model exists: {os.path.exists(SOURCE_MODEL_PATH)}')

## Step 6：辅助函数（伪标签生成、数据准备）

In [ ]:
# ----------------------------------------------------------------
# prepare_fold_data: 为一个 LOO fold 准备数据目录
#   - test_cycle:    原始真实标签，用于评估
#   - labeled_cycle: 前 label_ratio 比例的真实标签（半标签）
#   - 其余 7 个:     调用 pseudo.main() 生成伪标签（pseudo.py 原作者代码）
# 返回 (pseudo_dir, train_set) 供 train() 使用
# ----------------------------------------------------------------
def prepare_fold_data(
    source_model_path, target_temp, target_data_path,
    all_cycles, test_cycle, labeled_cycle,
    device, label_ratio=0.5, steps=90,
    pseudo_base='/tmp/tatn_pseudo'
):
    fold_id    = test_cycle.replace('.mat', '')
    pseudo_dir = os.path.join(pseudo_base, f'{target_temp}_test_{fold_id}')
    out_subdir = os.path.join(pseudo_dir, target_temp)
    os.makedirs(out_subdir, exist_ok=True)

    train_set = []  # file names for Mydataset (all except test_cycle)

    for cycle in all_cycles:
        orig_path = os.path.join(target_data_path, target_temp, f'{target_temp}{cycle}')
        mat = sio.loadmat(orig_path)
        cur = mat['current'].flatten()
        vol = mat['voltage'].flatten()
        tmp = mat['temp'].flatten()
        ah  = mat['ah'].reshape(-1, 1)

        if cycle == test_cycle:
            # 真实标签 — 仅供评估，不加入训练集
            n = len(cur) // steps * steps
            out_mat = dict(
                current=cur[:n].reshape(-1,1), voltage=vol[:n].reshape(-1,1),
                temp=tmp[:n].reshape(-1,1),    ah=ah[:n]
            )
            sio.savemat(os.path.join(out_subdir, f'{target_temp}{cycle}'), out_mat)

        elif cycle == labeled_cycle:
            # 半标签：只保留前 label_ratio 比例（对齐到 steps 整数倍）
            n_total   = len(cur) // steps * steps
            n_labeled = max(steps, int(n_total * label_ratio) // steps * steps)
            out_mat = dict(
                current=cur[:n_labeled].reshape(-1,1),
                voltage=vol[:n_labeled].reshape(-1,1),
                temp=tmp[:n_labeled].reshape(-1,1),
                ah=ah[:n_labeled]
            )
            sio.savemat(os.path.join(out_subdir, f'{target_temp}{cycle}'), out_mat)
            train_set.append(cycle)

        else:
            # 无标签 cycle — 调用 pseudo.main()（原作者代码，已修 bug）
            importlib.reload(pseudo)
            saved = pseudo.main(
                temp            = target_temp,
                model_path      = source_model_path,
                file            = cycle,
                save_dir        = pseudo_dir,
                data_path       = target_data_path,
                enable_fallback = False,  # strict reproduction: skip cycles with no valid segments
            )
            # validate: only keep pseudo files with >= steps(90) samples
            # short segments -> (0,3,90) in Mydataset -> empty tensor -> MAXLoss crash
            for fname in saved:
                fpath = os.path.join(pseudo_dir, target_temp, f'{target_temp}{fname}')
                try:
                    m_check = sio.loadmat(fpath)
                    if len(m_check['current']) >= steps:
                        train_set.append(fname)
                    else:
                        print(f'    skip {fname}: too short (<{steps} samples)')
                except Exception as e:
                    print(f'    skip {fname}: {e}')

    return pseudo_dir + '/', train_set


# ----------------------------------------------------------------
# make_transfer_models / make_transfer_optimizers
# 镜像 run.py 第 39-56 行，与原作者代码保持一致
# ----------------------------------------------------------------
def make_transfer_models(device):
    d = torch.device(device)
    models = {}
    models['conv']          = conv().to(d)
    models['lstm']          = lstm().to(d)
    models['fc']            = fc().to(d)
    models['regression']    = regression().to(d)
    models['conv_s']        = conv().to(d)
    models['lstm_s']        = lstm().to(d)
    models['fc_s']          = fc().to(d)
    models['regression_s']  = regression().to(d)
    models['discriminator'] = Discriminator().to(d)
    return models


def make_transfer_optimizers(models_dict, lr=1e-4):
    optimizers = {}
    optimizers['conv']          = optim.Adam(models_dict['conv'].parameters(),          lr=lr)
    optimizers['lstm']          = optim.Adam(models_dict['lstm'].parameters(),          lr=lr)
    optimizers['fc']            = optim.Adam(models_dict['fc'].parameters(),            lr=lr)
    optimizers['regression']    = optim.Adam(models_dict['regression'].parameters(),    lr=lr)
    optimizers['discriminator'] = optim.Adam(models_dict['discriminator'].parameters(), lr=lr)
    return optimizers


print('辅助函数定义完成')

## Step 7：运行半监督迁移（LOO × 全部目标温度）

对每个目标温度，进行 9 折留一法迁移训练，每折约 5–10 分钟（L4 GPU）。  
全部完成后自动保存结果到 Drive。

In [ ]:
import importlib, sys
os.chdir(WORK_DIR)

CYCLE_NAMES = [c.replace('.mat', '') for c in ALL_CYCLES]

# all_results[target_temp][cycle_name] = {'mae': float, 'rmse': float, 'max': float}
all_results = {t: {} for t in TARGET_TEMPS}

total_folds = len(TARGET_TEMPS) * len(ALL_CYCLES)
fold_counter = 0
exp_start = time.time()

for target_temp in TARGET_TEMPS:
    print(f'\n{"="*60}')
    print(f'  迁移方向: {SOURCE_TEMP}°C → {target_temp}°C')
    print(f'{"="*60}')

    for fold_i, test_cycle in enumerate(ALL_CYCLES):
        test_name     = test_cycle.replace('.mat', '')
        train_cycles  = [c for c in ALL_CYCLES if c != test_cycle]
        labeled_cycle = train_cycles[-1]   # 固定选最后一个 cycle 作为半标签
        fold_counter += 1

        print(f'\n  Fold {fold_i+1}/9  test={test_name}  labeled={labeled_cycle.replace(".mat","")}  [{fold_counter}/{total_folds}]')

        # --- 准备伪标签数据（调用 pseudo.main()，原作者代码）---
        pseudo_dir, pseudo_train_set = prepare_fold_data(
            SOURCE_MODEL_PATH, target_temp, TARGET_DATA_PATH,
            ALL_CYCLES, test_cycle, labeled_cycle,
            DEVICE, label_ratio=LABEL_RATIO,
        )

        # --- 重新加载模块（避免 importlib 缓存问题）---
        for m in ['models', 'mydata', 'train', 'utils']:
            if m in sys.modules:
                importlib.reload(sys.modules[m])
        from models import conv, lstm, fc, regression, Discriminator, load_saved_model
        from train import train as transfer_train

        # --- 初始化模型 ---
        models_dict = make_transfer_models(DEVICE)
        # 冻结源域 conv_s 参数
        for p in models_dict['conv_s'].parameters():
            p.requires_grad_(False)
        optimizers_dict = make_transfer_optimizers(models_dict, lr=LR)
        criterion = nn.MSELoss()

        run_tag = f'transfer_{SOURCE_TEMP}to{target_temp}_fold{fold_i}'

        # --- 运行迁移训练 ---
        t0 = time.time()
        min_mae, min_rmse, min_max = transfer_train(
            rundir          = run_tag,
            source_temp     = SOURCE_TEMP,
            target_temp     = target_temp,
            source_data_path= SOURCE_DATA_PATH,
            source_train_set= ALL_CYCLES,
            source_test_set = [ALL_CYCLES[0]],   # 仅评估第一条源域 cycle，节省时间
            target_data_path= pseudo_dir,
            target_train_set= pseudo_train_set,
            target_test_set = [test_cycle],
            models          = models_dict,
            criterion       = criterion,
            optimizers      = optimizers_dict,
            batch_size      = BATCH_SIZE,
            epochs          = EPOCHS,
            eval_interval   = EVAL_INTERVAL,
            lamb1           = LAMB1,
            lamb2           = LAMB2,
            lamb3           = LAMB3,
            seed            = SEED,
            device_type     = DEVICE,
            ifsave          = True,
            load_model      = True,
            model_path      = SOURCE_MODEL_PATH,
        )
        elapsed = (time.time() - t0) / 60

        # min_mae / min_rmse / min_max 是列表（每个 test cycle 对应一个元素，这里只有 1 个）
        all_results[target_temp][test_name] = {
            'mae':  min_mae[0]  if min_mae  else float('nan'),
            'rmse': min_rmse[0] if min_rmse else float('nan'),
            'max':  min_max[0]  if min_max  else float('nan'),
        }
        r = all_results[target_temp][test_name]
        print(f'  → MAE={r["mae"]*100:.3f}%  RMSE={r["rmse"]*100:.3f}%  MAX={r["max"]*100:.3f}%  ({elapsed:.1f} min)')

        # --- 保存中间结果到 Drive ---
        result_txt = os.path.join(DRIVE_TRANSFER, f'results_{SOURCE_TEMP}to{target_temp}.txt')
        with open(result_txt, 'a') as f:
            f.write(f'{test_name}  mae={r["mae"]:.6f}  rmse={r["rmse"]:.6f}  max={r["max"]:.6f}\n')

    total_elapsed = (time.time() - exp_start) / 60
    print(f'\n[{SOURCE_TEMP}°C → {target_temp}°C] 完成  总用时: {total_elapsed:.1f} min')

print('\n全部迁移实验完成')

## Step 8：汇总结果，对照论文 Table IV

In [ ]:
# 论文 Table IV 参考值（TATN 半监督，源域=10°C）
# 每列 = 目标温度，行对应 cycle（顺序：Cycle1-4, NN, US06, HWFET, LA92, UDDS）
PAPER_SEMISUP = {
    # 来自 Table IV：TATN MAE (%)，若无精确数值可留空
    '25':  None,   # 填入论文对应列均值（如有）
    '0':   None,
    'n10': None,
    'n20': None,
}

print(f'半监督迁移结果  源域: {SOURCE_TEMP}°C\n')
header = f'{"Cycle":<10}'
for t in TARGET_TEMPS:
    header += f'  {t}°C MAE%  RMSE%'
print(header)
print('-' * (10 + len(TARGET_TEMPS) * 20))

avg_mae  = {t: [] for t in TARGET_TEMPS}
avg_rmse = {t: [] for t in TARGET_TEMPS}

for cycle_name in CYCLE_NAMES:
    row = f'{cycle_name:<10}'
    for t in TARGET_TEMPS:
        r = all_results[t].get(cycle_name, {})
        mae  = r.get('mae',  float('nan'))
        rmse = r.get('rmse', float('nan'))
        row += f'  {mae*100:7.3f}   {rmse*100:7.3f}'
        if not (mae != mae):  # not NaN
            avg_mae[t].append(mae)
            avg_rmse[t].append(rmse)
    print(row)

print('-' * (10 + len(TARGET_TEMPS) * 20))
avg_row = f'{"AVG":<10}'
for t in TARGET_TEMPS:
    m  = sum(avg_mae[t])  / len(avg_mae[t])  if avg_mae[t]  else float('nan')
    rm = sum(avg_rmse[t]) / len(avg_rmse[t]) if avg_rmse[t] else float('nan')
    avg_row += f'  {m*100:7.3f}   {rm*100:7.3f}'
print(avg_row)

print('\n论文 Table IV 参考（TATN 半监督）:')
print('  源域 10°C 各目标温度 MAE 均值（%）：约 1.XX%（请对照论文原表填入）')

## Step 9（可选）：单独打印某个温度对的详细结果

In [ ]:
# 修改 target_temp_show 查看不同温度对的结果
target_temp_show = '25'

print(f'\n{SOURCE_TEMP}°C → {target_temp_show}°C  半监督迁移详细结果')
print(f'{"Drive Cycle":<12}  {"MAE (%)":>9}  {"RMSE (%)":>9}  {"MAX (%)":>9}')
print('-' * 46)
maes, rmses = [], []
for cycle_name in CYCLE_NAMES:
    r = all_results.get(target_temp_show, {}).get(cycle_name, {})
    mae  = r.get('mae',  float('nan'))
    rmse = r.get('rmse', float('nan'))
    mx   = r.get('max',  float('nan'))
    print(f'{cycle_name:<12}  {mae*100:9.3f}  {rmse*100:9.3f}  {mx*100:9.3f}')
    if not (mae != mae):
        maes.append(mae); rmses.append(rmse)
print('-' * 46)
if maes:
    print(f'{"AVG":<12}  {sum(maes)/len(maes)*100:9.3f}  {sum(rmses)/len(rmses)*100:9.3f}')

## Step 10（可选）：保存完整结果到 Drive

In [ ]:
import json

# 保存 all_results 为 JSON（方便后续读取）
result_json = os.path.join(DRIVE_TRANSFER, f'all_results_{SOURCE_TEMP}to_all.json')
with open(result_json, 'w') as f:
    json.dump(all_results, f, indent=2)
print(f'结果已保存: {result_json}')

# 保存可读的汇总表格
summary_path = os.path.join(DRIVE_TRANSFER, f'summary_{SOURCE_TEMP}to_all.txt')
with open(summary_path, 'w') as f:
    f.write(f'TATN 半监督迁移结果  源域: {SOURCE_TEMP}°C\n\n')
    f.write(f'{"Cycle":<10}')
    for t in TARGET_TEMPS:
        f.write(f'  {t}°C_MAE%  {t}°C_RMSE%')
    f.write('\n' + '-'*(10 + len(TARGET_TEMPS)*20) + '\n')
    for cycle_name in CYCLE_NAMES:
        row = f'{cycle_name:<10}'
        for t in TARGET_TEMPS:
            r = all_results[t].get(cycle_name, {})
            row += f'  {r.get("mae",float("nan"))*100:9.3f}  {r.get("rmse",float("nan"))*100:9.3f}'
        f.write(row + '\n')
    f.write('-'*(10 + len(TARGET_TEMPS)*20) + '\n')
    avg_row = f'{"AVG":<10}'
    for t in TARGET_TEMPS:
        valid = [v['mae'] for v in all_results[t].values() if 'mae' in v]
        validr= [v['rmse'] for v in all_results[t].values() if 'rmse' in v]
        avg_row += f'  {sum(valid)/len(valid)*100 if valid else float("nan"):9.3f}  {sum(validr)/len(validr)*100 if validr else float("nan"):9.3f}'
    f.write(avg_row + '\n')
print(f'汇总表格已保存: {summary_path}')